In [32]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [33]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "Extracted_Data"

OUTPUT_DIR = PROJECT_DIR / "processed"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Data Folder :", DATA_DIR)

Data Folder : c:\Users\vikhy\Desktop\Nikitha project\Extracted_Data


In [34]:
excel_files = sorted(DATA_DIR.rglob("*.xlsx"))

print(f"Total Excel Files : {len(excel_files)}")

excel_files[:5]

Total Excel Files : 24


[WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Glauburgstraße 50.1270654,8.6893089/2024-10-08_to_2024-10-13_UI_202403B0923.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Hanauer Landstraße 50.1125629,8.7039647/2024-10-08_to_2024-10-13_UI_202310B0614.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Hanauer Landstraße 50.1134021,8.6969654/2024-10-08_to_2024-10-13_UI_202403B0924.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Töngesgasse 50.1135489,8.6857431/2024-10-08_to_2024-10-13_UI_202403B0908.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 09.09-15.09/DZwEI 09.09-15.09/B44/2024-09-09_to_2024-09-15_UI_202403B0908.xlsx')]

In [35]:
IGNORE_SHEETS = [
    "Erklärungen"
]

In [36]:
VEHICLE_CATEGORIES = [
    "Gesamt-Kfz",
    "SV >3.5t",
    "LV <3.5t",
    "Pkw",
    "Pkw mit Anhänger",
    "Lkw",
    "Lkw mit Anhänger",
    "Bus",
    "Lieferwagen",
    "Kraftrad",
    "Sattel-Kfz"
]

In [37]:
def extract_metadata(filepath):

    week = filepath.parents[1].name

    folder = filepath.parent.name

    latitude = np.nan
    longitude = np.nan

    match = re.search(
        r'([-+]?\d+\.\d+),([-+]?\d+\.\d+)',
        folder
    )

    if match:

        latitude = float(match.group(1))
        longitude = float(match.group(2))

        folder = folder.replace(match.group(0),"").strip()

    return {

        "Week": week,

        "Measurement_Site": folder,

        "Latitude": latitude,

        "Longitude": longitude,

        "Source_File": filepath.name

    }

In [38]:
def read_sheet(file, sheet):
    """
    Read only the traffic table.
    """

    df = pd.read_excel(
        file,
        sheet_name=sheet,
        header=11,
        nrows=96
    )

    return df

In [39]:
sample = excel_files[0]

df = read_sheet(sample, "Gesamt-Kfz")

print(df.shape)

df.head()

(96, 23)


,Startzeit,von Unten nach Oben,von Links nach Oben,von Links nach Unten,von Oben nach Unten,Unnamed: 5,Unnamed: 6,Intervall,Summe Stunde,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Fahrtrichtung Süd-West,Fahrtrichtung Nord-Ost,Q,Fahrtrichtung Nord-West,Fahrtrichtung Süd-Ost,Q.1,Fahrtrichtung Nord-Ost.1,Fahrtrichtung Süd-West.1,Q.2
0,00:00:00,4,0,1,6,NaN,NaN,11,0,NaN,NaN,NaN,NaN,NaN,4,7,0,1,0,0,6,4,0
1,00:15:00,2,0,1,3,NaN,NaN,6,0,NaN,NaN,NaN,NaN,NaN,2,4,0,1,0,0,3,2,0
2,00:30:00,1,0,2,3,NaN,NaN,6,0,NaN,NaN,NaN,NaN,NaN,1,5,0,2,0,0,3,1,0
3,00:45:00,3,0,1,3,NaN,NaN,7,0,NaN,NaN,NaN,NaN,NaN,3,4,0,1,0,0,3,3,0
4,01:00:00,4,1,0,0,NaN,NaN,5,0,NaN,NaN,NaN,NaN,NaN,4,0,0,1,0,0,0,5,0


In [40]:
def remove_empty_columns(df):

    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    return df

In [41]:
REMOVE_COLUMNS = [

    "Intervall",

    "Summe Stunde",

    "Q",

    "Q.1",

    "Q.2"
]

In [42]:
def remove_summary_columns(df):

    cols = [

        c

        for c in df.columns

        if c not in REMOVE_COLUMNS

    ]

    return df[cols]

In [43]:
def rename_columns(df):

    df = df.rename(

        columns={

            "Startzeit":"Time"

        }

    )

    return df

In [44]:
test = read_sheet(sample,"Gesamt-Kfz")

test = remove_empty_columns(test)

test = remove_summary_columns(test)

test = rename_columns(test)

test.head()

,Time,von Unten nach Oben,von Links nach Oben,von Links nach Unten,von Oben nach Unten,Fahrtrichtung Süd-West,Fahrtrichtung Nord-Ost,Fahrtrichtung Nord-West,Fahrtrichtung Süd-Ost,Fahrtrichtung Nord-Ost.1,Fahrtrichtung Süd-West.1
0,00:00:00,4,0,1,6,4,7,1,0,6,4
1,00:15:00,2,0,1,3,2,4,1,0,3,2
2,00:30:00,1,0,2,3,1,5,2,0,3,1
3,00:45:00,3,0,1,3,3,4,1,0,3,3
4,01:00:00,4,1,0,0,4,0,1,0,0,5


In [45]:
print(test.columns.tolist())

['Time', 'von Unten nach Oben', 'von Links nach Oben', 'von Links nach Unten', 'von Oben nach Unten', 'Fahrtrichtung Süd-West', 'Fahrtrichtung Nord-Ost', 'Fahrtrichtung Nord-West', 'Fahrtrichtung Süd-Ost', 'Fahrtrichtung Nord-Ost.1', 'Fahrtrichtung Süd-West.1']


In [46]:
test.columns.tolist()

['Time',
 'von Unten nach Oben',
 'von Links nach Oben',
 'von Links nach Unten',
 'von Oben nach Unten',
 'Fahrtrichtung Süd-West',
 'Fahrtrichtung Nord-Ost',
 'Fahrtrichtung Nord-West',
 'Fahrtrichtung Süd-Ost',
 'Fahrtrichtung Nord-Ost.1',
 'Fahrtrichtung Süd-West.1']

In [47]:
def clean_sheet(df):
    """
    Clean a traffic sheet by:
    - Removing empty/helper columns
    - Renaming Time column
    """

    cols_to_drop = [
        c for c in df.columns
        if (
            str(c).startswith("Unnamed")
            or "Intervall" in str(c)
            or "Summe" in str(c)
            or str(c).startswith("Q")
        )
    ]

    df = df.drop(columns=cols_to_drop, errors="ignore")

    df = df.rename(
        columns={
            "Startzeit": "Time"
        }
    )

    return df

In [48]:
def process_sheet(filepath, sheet):

    metadata = extract_metadata(filepath)

    df = read_sheet(filepath, sheet)

    df = clean_sheet(df)

    df.insert(0, "Vehicle_Category", sheet)

    df.insert(0, "Source_File", metadata["Source_File"])

    df.insert(0, "Longitude", metadata["Longitude"])

    df.insert(0, "Latitude", metadata["Latitude"])

    df.insert(0, "Measurement_Site", metadata["Measurement_Site"])

    df.insert(0, "Week", metadata["Week"])

    return df

In [49]:
sample = excel_files[0]

test = process_sheet(
    sample,
    "Gesamt-Kfz"
)

print(test.shape)

test.head()

(96, 17)


,Week,Measurement_Site,Latitude,Longitude,Source_File,Vehicle_Category,Time,von Unten nach Oben,von Links nach Oben,von Links nach Unten,von Oben nach Unten,Fahrtrichtung Süd-West,Fahrtrichtung Nord-Ost,Fahrtrichtung Nord-West,Fahrtrichtung Süd-Ost,Fahrtrichtung Nord-Ost.1,Fahrtrichtung Süd-West.1
0,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:00:00,4,0,1,6,4,7,1,0,6,4
1,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:15:00,2,0,1,3,2,4,1,0,3,2
2,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:30:00,1,0,2,3,1,5,2,0,3,1
3,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:45:00,3,0,1,3,3,4,1,0,3,3
4,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,01:00:00,4,1,0,0,4,0,1,0,0,5


In [50]:
master_data = []

for file in tqdm(excel_files):

    for sheet in VEHICLE_CATEGORIES:

        try:

            temp = process_sheet(file, sheet)

            master_data.append(temp)

        except Exception as e:

            print(f"Skipped {file.name} -> {sheet}")

            print(e)

  0%|          | 0/24 [00:00<?, ?it/s]

 21%|██        | 5/24 [00:01<00:07,  2.66it/s]

Skipped 2024-09-09_to_2024-09-15_UI_202403B0908.xlsx -> Sattel-Kfz
Worksheet named 'Sattel-Kfz' not found


 25%|██▌       | 6/24 [00:02<00:06,  2.81it/s]

Skipped 2024-09-09_to_2024-09-15_UI_202403B0924.xlsx -> Sattel-Kfz
Worksheet named 'Sattel-Kfz' not found


...

 92%|█████████▏| 22/24 [00:07<00:00,  2.78it/s]

Skipped 2024-09-30_to_2024-10-07_UI_202403B0923.xlsx -> SV >3.5t
Worksheet named 'SV >3.5t' not found
Skipped 2024-09-30_to_2024-10-07_UI_202403B0923.xlsx -> LV <3.5t
Worksheet named 'LV <3.5t' not found


 96%|█████████▌| 23/24 [00:08<00:00,  2.81it/s]

Skipped 2024-09-30_to_2024-10-07_UI_202310B0614.xlsx -> SV >3.5t
Worksheet named 'SV >3.5t' not found
Skipped 2024-09-30_to_2024-10-07_UI_202310B0614.xlsx -> LV <3.5t
Worksheet named 'LV <3.5t' not found


100%|██████████| 24/24 [00:08<00:00,  2.81it/s]


In [51]:
master_df = pd.concat(
    master_data,
    ignore_index=True
)

print(master_df.shape)

master_df.head()

(23424, 43)


,Week,Measurement_Site,Latitude,Longitude,Source_File,Vehicle_Category,Time,von Unten nach Oben,von Links nach Oben,von Links nach Unten,von Oben nach Unten,Fahrtrichtung Süd-West,Fahrtrichtung Nord-Ost,Fahrtrichtung Nord-West,Fahrtrichtung Süd-Ost,Fahrtrichtung Nord-Ost.1,Fahrtrichtung Süd-West.1,von Rechts nach Unten,von Rechts nach Oben,Von Links nach Oben,Von Links nach Unten,Von Oben nach Unten,Fahrtrichtung Süd,Fahrtrichtung Nord,Fahrtrichtung West,Fahrtrichtung Ost,Leicht-rechts in Links,Rechts in Oben,Geradeaus in Links,Rechts in Oben.1,Falschfahrer,Falschfahrer.1,Falschfahrer.2,Geradeaus in Unten,Geradeaus in Oben,Fahrtrichtung Nord.1,Fahrtrichtung Süd.1,von Rechts Links nach Unten,von Links Geradeaus in Rechts,von Oben Links in Rechts,von Oben Geradeaus in Unten,von Unten Geradeaus in Oben,nach Unten
0,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:00:00,4.0,0.0,1.0,6.0,4.0,7.0,1.0,0.0,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:15:00,2.0,0.0,1.0,3.0,2.0,4.0,1.0,0.0,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:30:00,1.0,0.0,2.0,3.0,1.0,5.0,2.0,0.0,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:45:00,3.0,0.0,1.0,3.0,3.0,4.0,1.0,0.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,01:00:00,4.0,1.0,0.0,0.0,4.0,0.0,1.0,0.0,0.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
print(master_df.columns.tolist())

['Week', 'Measurement_Site', 'Latitude', 'Longitude', 'Source_File', 'Vehicle_Category', 'Time', 'von Unten nach Oben', 'von Links nach Oben', 'von Links nach Unten', 'von Oben nach Unten', 'Fahrtrichtung Süd-West', 'Fahrtrichtung Nord-Ost', 'Fahrtrichtung Nord-West', 'Fahrtrichtung Süd-Ost', 'Fahrtrichtung Nord-Ost.1', 'Fahrtrichtung Süd-West.1', 'von Rechts nach Unten', 'von Rechts nach Oben', 'Von Links nach Oben', 'Von Links nach Unten', 'Von Oben nach Unten', 'Fahrtrichtung Süd', 'Fahrtrichtung Nord', 'Fahrtrichtung West', 'Fahrtrichtung Ost', 'Leicht-rechts in Links', 'Rechts in Oben', 'Geradeaus in Links', 'Rechts in Oben.1', 'Falschfahrer', 'Falschfahrer.1', 'Falschfahrer.2', 'Geradeaus in Unten', 'Geradeaus in Oben', 'Fahrtrichtung Nord.1', 'Fahrtrichtung Süd.1', 'von Rechts Links nach Unten', 'von Links Geradeaus in Rechts', 'von Oben Links in Rechts', 'von Oben Geradeaus in Unten', 'von Unten Geradeaus in Oben', 'nach Unten']


In [53]:
metadata_cols = [

    "Week",

    "Measurement_Site",

    "Latitude",

    "Longitude",

    "Source_File",

    "Vehicle_Category",

    "Time"

]

traffic_cols = [

    c

    for c in master_df.columns

    if c not in metadata_cols

]

for col in traffic_cols:

    master_df[col] = pd.to_numeric(

        master_df[col],

        errors="coerce"

    )

In [54]:
master_df["Total_Traffic"] = (

    master_df[traffic_cols]

    .sum(axis=1)
)

In [55]:
print("="*60)

print("Rows :",len(master_df))

print("Columns :",len(master_df.columns))

print()

print(master_df.isnull().sum())

Rows : 23424
Columns : 44

Week                                 0
Measurement_Site                     0
Latitude                         14976
Longitude                        14976
Source_File                          0
Vehicle_Category                     0
Time                                 0
von Unten nach Oben              19200
von Links nach Oben              19200
von Links nach Unten             19200
von Oben nach Unten              17088
Fahrtrichtung Süd-West           11328
Fahrtrichtung Nord-Ost           11328
Fahrtrichtung Nord-West          15168
Fahrtrichtung Süd-Ost            15168
Fahrtrichtung Nord-Ost.1         15552
Fahrtrichtung Süd-West.1         15552
von Rechts nach Unten            21312
von Rechts nach Oben             21312
Von Links nach Oben              21312
Von Links nach Unten             21312
Von Oben nach Unten              21312
Fahrtrichtung Süd                10560
Fahrtrichtung Nord               10560
Fahrtrichtung West               1766

In [56]:
master_df.sample(10)

,Week,Measurement_Site,Latitude,Longitude,Source_File,Vehicle_Category,Time,von Unten nach Oben,von Links nach Oben,von Links nach Unten,von Oben nach Unten,Fahrtrichtung Süd-West,Fahrtrichtung Nord-Ost,Fahrtrichtung Nord-West,Fahrtrichtung Süd-Ost,Fahrtrichtung Nord-Ost.1,Fahrtrichtung Süd-West.1,von Rechts nach Unten,von Rechts nach Oben,Von Links nach Oben,Von Links nach Unten,Von Oben nach Unten,Fahrtrichtung Süd,Fahrtrichtung Nord,Fahrtrichtung West,Fahrtrichtung Ost,Leicht-rechts in Links,Rechts in Oben,Geradeaus in Links,Rechts in Oben.1,Falschfahrer,Falschfahrer.1,Falschfahrer.2,Geradeaus in Unten,Geradeaus in Oben,Fahrtrichtung Nord.1,Fahrtrichtung Süd.1,von Rechts Links nach Unten,von Links Geradeaus in Rechts,von Oben Links in Rechts,von Oben Geradeaus in Unten,von Unten Geradeaus in Oben,nach Unten,Total_Traffic
14477,DZwEI 16.09-23.09,Heiligkreuzgasse,NaN,NaN,2024-09-16_to_2024-09-23_UI_202403B0923.xlsx,LV <3.5t,19:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,0.0,0.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,48.0
6203,DZwEI 09.09-15.09,Heiligkreuzgasse,NaN,NaN,2024-09-09_to_2024-09-15_UI_202403B0923.xlsx,Gesamt-Kfz,14:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.0,0.0,0.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN,105.0
13847,DZwEI 16.09-23.09,Habsburgerallee 59,NaN,NaN,2024-09-16_to_2024-09-23_UI_202403B0924.xlsx,Lkw mit Anhänger,05:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0
23392,DZwEI 30.09-07.10,Richard Wagner Straße,NaN,NaN,2024-09-30_to_2024-10-07_UI_202310B0614.xlsx,Sattel-Kfz,16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,2.0
15735,DZwEI 16.09-23.09,Mainzer Landstraße 32,NaN,NaN,2024-09-16_to_2024-09-23_UI_202310B0614.xlsx,Lkw,21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0
13235,DZwEI 16.09-23.09,B44,NaN,NaN,2024-09-16_to_2024-09-23_UI_202403B0908.xlsx,Kraftrad,20:45:00,NaN,NaN,NaN,NaN,0.0,0.0,2.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0
19949,DZwEI 30.09-07.10,Friedberger Tor,NaN,NaN,2024-09-30_to_2024-10-07_UI_202403B0924.xlsx,SV >3.5t,19:15:00,NaN,NaN,NaN,NaN,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3.0,0.0,NaN,NaN,9.0
3366,DZwEI 08.10-13.10,Töngesgasse,50.113549,8.685743,2024-10-08_to_2024-10-13_UI_202403B0908.xlsx,LV <3.5t,01:30:00,0.0,0.0,1.0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
7511,DZwEI 09.09-15.09,Mainzer Landstraße 32,NaN,NaN,2024-09-09_to_2024-09-15_UI_202310B0614.xlsx,Pkw mit Anhänger,05:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
19650,DZwEI 24.09-29.09,Richard Wagner Straße,NaN,NaN,2024-09-24_to_2024-09-29_UI_202310B0614.xlsx,Kraftrad,16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,1.0,2.0


In [57]:
master_df.shape

(23424, 44)

In [58]:
master_df.columns.tolist()

['Week',
 'Measurement_Site',
 'Latitude',
 'Longitude',
 'Source_File',
 'Vehicle_Category',
 'Time',
 'von Unten nach Oben',
 'von Links nach Oben',
 'von Links nach Unten',
 'von Oben nach Unten',
 'Fahrtrichtung Süd-West',
 'Fahrtrichtung Nord-Ost',
 'Fahrtrichtung Nord-West',
 'Fahrtrichtung Süd-Ost',
 'Fahrtrichtung Nord-Ost.1',
 'Fahrtrichtung Süd-West.1',
 'von Rechts nach Unten',
 'von Rechts nach Oben',
 'Von Links nach Oben',
 'Von Links nach Unten',
 'Von Oben nach Unten',
 'Fahrtrichtung Süd',
 'Fahrtrichtung Nord',
 'Fahrtrichtung West',
 'Fahrtrichtung Ost',
 'Leicht-rechts in Links',
 'Rechts in Oben',
 'Geradeaus in Links',
 'Rechts in Oben.1',
 'Falschfahrer',
 'Falschfahrer.1',
 'Falschfahrer.2',
 'Geradeaus in Unten',
 'Geradeaus in Oben',
 'Fahrtrichtung Nord.1',
 'Fahrtrichtung Süd.1',
 'von Rechts Links nach Unten',
 'von Links Geradeaus in Rechts',
 'von Oben Links in Rechts',
 'von Oben Geradeaus in Unten',
 'von Unten Geradeaus in Oben',
 'nach Unten',
 'Total

In [59]:
master_df.head()

,Week,Measurement_Site,Latitude,Longitude,Source_File,Vehicle_Category,Time,von Unten nach Oben,von Links nach Oben,von Links nach Unten,von Oben nach Unten,Fahrtrichtung Süd-West,Fahrtrichtung Nord-Ost,Fahrtrichtung Nord-West,Fahrtrichtung Süd-Ost,Fahrtrichtung Nord-Ost.1,Fahrtrichtung Süd-West.1,von Rechts nach Unten,von Rechts nach Oben,Von Links nach Oben,Von Links nach Unten,Von Oben nach Unten,Fahrtrichtung Süd,Fahrtrichtung Nord,Fahrtrichtung West,Fahrtrichtung Ost,Leicht-rechts in Links,Rechts in Oben,Geradeaus in Links,Rechts in Oben.1,Falschfahrer,Falschfahrer.1,Falschfahrer.2,Geradeaus in Unten,Geradeaus in Oben,Fahrtrichtung Nord.1,Fahrtrichtung Süd.1,von Rechts Links nach Unten,von Links Geradeaus in Rechts,von Oben Links in Rechts,von Oben Geradeaus in Unten,von Unten Geradeaus in Oben,nach Unten,Total_Traffic
0,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:00:00,4.0,0.0,1.0,6.0,4.0,7.0,1.0,0.0,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0
1,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:15:00,2.0,0.0,1.0,3.0,2.0,4.0,1.0,0.0,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0
2,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:30:00,1.0,0.0,2.0,3.0,1.0,5.0,2.0,0.0,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0
3,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,00:45:00,3.0,0.0,1.0,3.0,3.0,4.0,1.0,0.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0
4,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Gesamt-Kfz,01:00:00,4.0,1.0,0.0,0.0,4.0,0.0,1.0,0.0,0.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0


In [60]:
traffic_cols = [

    c

    for c in master_df.columns

    if c not in [

        "Week",

        "Measurement_Site",

        "Latitude",

        "Longitude",

        "Source_File",

        "Vehicle_Category",

        "Time"

    ]

]

master_df[traffic_cols] = master_df[traffic_cols].fillna(0)

In [61]:
master_df["Total_Traffic"] = master_df[traffic_cols].sum(axis=1)

In [62]:
master_df = master_df.sort_values(

    [

        "Week",

        "Measurement_Site",

        "Vehicle_Category",

        "Time"

    ]

).reset_index(drop=True)

In [63]:
master_df.to_csv(

    OUTPUT_DIR/"master1_dataset.csv",

    index=False

)

print("Master Dataset Saved")

Master Dataset Saved
